In [1]:
import numpy as np
from time import time_ns
from pynq import Overlay, allocate

In [22]:
NSTEPS     = 5
CHUNK_SIZE = 8           # next_pow2(NSTEPS) — matches HLS
M, N       = 1080,1920      # bump to 1080, 1920 when you rebuild
MN         = M * N
DIR = f"TestData_{M}x{N}"

# Control register offsets (from HLS: <proj>/syn/report/*_drv.h)
CTRL_REG = 0x00          # bit0=ap_start, bit1=ap_done, bit2=ap_idle, bit3=ap_ready


In [23]:
overlay = Overlay(f"{DIR}/equalStep.bit",download=True)
kernel  = overlay.equalStep_baseline_0
dma     = overlay.axi_dma_0

_in_buf = allocate(shape=(MN * CHUNK_SIZE,), dtype=np.uint16)
_out_buf= allocate(shape=(MN,),dtype=np.uint64)
_in_view= np.asarray(_in_buf).reshape(MN,CHUNK_SIZE)

In [20]:
def run_kernel(imStack):
    """
    imStack : (NSTEPS, M, N) uint16 numpy array
    returns : (phase, mod) — each (M, N) float64
    """
    assert imStack.shape == (NSTEPS, M, N)
    assert imStack.dtype == np.uint16

    # Pack one pixel per AXIS beat: lanes [0..NSTEPS) = phase-shifted samples,
    # lanes [NSTEPS..CHUNK_SIZE) = 0 padding. Shape ends up (MN, 8) = 128 bits/beat.
    _in_view[:, :NSTEPS] = imStack.reshape(NSTEPS, MN).T     # (MN, NSTEPS)

#     print("in_buf.nbytes  :", in_buf.nbytes)
#     print("buf reg width  :", dma.sendchannel._max_size)
    # Start kernel, then DMAs. S2MM (output) first so it's ready to receive.
    start = time_ns()
    kernel.write(CTRL_REG, 0x01)               # ap_start
    dma.recvchannel.transfer(_out_buf)
    dma.sendchannel.transfer(_in_buf)
    
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    elapsed = (time_ns() - start)/1000000
    print(f"Ran in {elapsed}ms")
#     print("send idle :", dma.sendchannel.idle)
#     print("recv idle :", dma.recvchannel.idle)
#     print("ap_ctrl   :", hex(kernel.read(0x00)))
#     print("s2mm_SR   :", hex(dma.recvchannel._mmio.read(0x34)))   # S2MM status
#     print("mm2s_SR   :", hex(dma.sendchannel._mmio.read(0x04)))   # MM2S status
#     print("out_buf nonzero count:", np.count_nonzero(np.asarray(out_buf)))
#     # pixels at (0,0) and at (row 8, col 0) — mod differs here (318.88 vs 319.27)
#     print(f"out_buf[0]   (pixel 0,0)   : {out_buf[0]:016x}")
#     print(f"out_buf[8*64] (pixel 8,0)   : {out_buf[8*64]:016x}")
#     print(f"out_buf[16*64] (pixel 16,0) : {out_buf[16*64]:016x}")
    
    
#     for i in range(8):
#         print(f"out_buf[{i}]: {out_buf[i]:016x}")

    # Poll ap_done (optional — S2MM completion already implies it)
#     while (kernel.read(CTRL_REG) & 0x2) == 0:
#         pass

    # Unpack the packed struct on the 64-bit output bus.
    # Layout from HLS (LSB-first field order of phase_mod_t):
    #   bits [19:0]  = wrappedPhase  — ap_fixed<20,4>, 16 frac bits, signed
    #   bits [43:20] = mod           — ap_fixed<24,18>, 6 frac bits,  signed
    raw = np.asarray(_out_buf).astype(np.int64, copy=False)

    phase_raw = (raw << 44) >> 44
    phase = phase_raw.astype(np.float64) * (1.0 / (1 << 16))

    mod_raw = (raw << 12) >> 32
    mod = mod_raw.astype(np.float64) * (1.0 / (1 << 10))
    return phase.reshape(M, N), mod.reshape(M, N)

In [16]:
STACK_COUNT = 5
STACKS_TEST = [0,1,2,3,4]

In [24]:
# Load your 5 PNGs into a (5, M, N) uint16 stack
from PIL import Image
import os
if os.path.isfile(f"{DIR}/imStacks_{M}x{N}.npy") and os.path.isfile(f"{DIR}/phases_golden_{M}x{N}.npy") and os.path.isfile(f"{DIR}/mods_golden_{M}x{N}.npy"):
    imStacks = np.load(f"{DIR}/imStacks_{M}x{N}.npy")
    phases_golden = np.load(f"{DIR}/phases_golden_{M}x{N}.npy")
    mods_golden = np.load(f"{DIR}/mods_golden_{M}x{N}.npy")
else:
    imStacks = [np.stack([np.array(Image.open(f"{DIR}/stack_{s}_{k}.png"), dtype=np.uint16)for k in range(NSTEPS)])for s in range(STACK_COUNT)]
    phases_golden = [np.loadtxt(f"{DIR}/phi_{s}.csv",delimiter=",",dtype=np.float64)for s in range(STACK_COUNT)]
    mods_golden = [np.loadtxt(f"{DIR}/mod_{s}.csv",delimiter=",",dtype=np.float64)for s in range(STACK_COUNT)]

    np.save(f"{DIR}/imStacks_{M}x{N}.npy",imStacks)
    np.save(f"{DIR}/phases_golden_{M}x{N}.npy",phases_golden)
    np.save(f"{DIR}/mods_golden_{M}x{N}.npy",mods_golden)

In [18]:
def get_outp_stats(test,gold):
    diff = test-gold
    print(f"Max: {np.max(diff)}\tMin: {np.min(diff)}\tMean: {np.mean(diff)}\tRMS: {np.sqrt(np.mean(diff**2))}")

In [25]:
for stack in STACKS_TEST:
    imStack = imStacks[stack]
    phase_golden = phases_golden[stack]
    mod_golden = mods_golden[stack]
    print(f"---------------------------------------- STACK {stack} ----------------------------------------\n\n")
    start=time_ns()
    phase, mod = run_kernel(imStack)
    elapsed = (time_ns() - start)/1000000
    print(f"run_kernel in {elapsed}ms")
    print("phase range:", phase.min(), phase.max())
    print("mod   range:", mod.min(),   mod.max())

    print("Phase Error:")
    get_outp_stats(phase,phase_golden)

    print("Mod Error:")
    get_outp_stats(mod,mod_golden)

    phase_good = np.allclose(phase,phase_golden,0,0.0001)
    mod_good = np.allclose(mod,mod_golden,0,0.01)

    if phase_good:
        print("Phases CORRECT")
    else:
        print("Phases INCORRECT")

    if mod_good:
        print("Modulation CORRECT")
    else:
        print("Modulation INCORRECT")
    print(f"\n\n-----------------------------------------------------------------------------------------")

---------------------------------------- STACK 0 ----------------------------------------


Ran in 67.518041ms
run_kernel in 6456.763919ms
phase range: -3.13677978515625 3.1367950439453125
mod   range: 318.5234375 321.2763671875
Phase Error:
Max: 1.645259828508827e-05	Min: -3.264497537003308e-05	Mean: -1.2576026618195052e-05	RMS: 1.7874564868399436e-05
Mod Error:
Max: 0.006657584845015663	Min: -0.004847036064006716	Mean: 0.002589788489628962	RMS: 0.004253993963265953
Phases CORRECT
Modulation CORRECT


-----------------------------------------------------------------------------------------
---------------------------------------- STACK 1 ----------------------------------------


Ran in 161.014142ms
run_kernel in 3608.406653ms
phase range: -3.13677978515625 3.1367950439453125
mod   range: 318.5234375 321.2763671875
Phase Error:
Max: 1.737000017021373e-05	Min: -3.264497537003308e-05	Mean: -5.77293869455416e-06	RMS: 1.4462345654313562e-05
Mod Error:
Max: 0.006657584845015663	Min: -0.005